In [ ]:
--What is the distribution of patients by age and gender?--

SELECT
gender,
CASE
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 0 AND 17 THEN 'Pediatric'
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 18 AND 64 THEN 'Adult'
ELSE 'Senior'
END AS age_group,
COUNT(*) AS patient_count
FROM [Healthcare_Database].[dbo].Patients
GROUP BY
gender,
CASE
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 0 AND 17 THEN 'Pediatric'
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 18 AND 64 THEN 'Adult'
ELSE 'Senior'
END
ORDER BY patient_count Desc

(6 rows affected)

gender | age_group | patient_count
-------+-----------+--------------
Female | Adult     | 44           
Male   | Adult     | 26           
Male   | Senior    | 14           
Female | Senior    | 7            
Female | Pediatric | 6            
Male   | Pediatric | 3            
(6 rows)

Total execution time: 00:00:00.012

In [ ]:
--How do diagnoses differ by age group and gender?--

SELECT
gender,
diagnosis,
CASE
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 0 AND 17 THEN 'Pediatric'
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 18 AND 64 THEN 'Adult'
ELSE 'Senior'
END AS age_group,
COUNT(*) AS patient_count
FROM [Healthcare_Database].[dbo].Patients AS p
INNER JOIN [Healthcare_Database].[dbo].[Outpatient Visits] AS ov
ON p.patient_id = ov.patient_id
WHERE diagnosis <> 'Unknown'
GROUP BY
gender,
diagnosis,
CASE
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 0 AND 17 THEN 'Pediatric'
WHEN DATEDIFF(year, date_of_birth, GETDATE()) BETWEEN 18 AND 64 THEN 'Adult'
ELSE 'Senior'
END
ORDER BY patient_count DESC, diagnosis ASC

(44 rows affected)

gender | diagnosis             | age_group | patient_count
-------+-----------------------+-----------+--------------
Female | Diabetes              | Adult     | 20           
Female | Hypertension          | Adult     | 13           
Female | Muscle Injury         | Adult     | 12           
Female | Respiratory Illness   | Adult     | 12           
Male   | Diabetes              | Adult     | 11           
Male   | Hypertension          | Adult     | 9            
Female | Hyperlipidemia        | Adult     | 7            
Male   | Muscle Injury         | Adult     | 6            
Male   | Respiratory Illness   | Senior    | 6            
Female | Migraine              | Adult     | 5            
Male   | Allergic Reaction     | Adult     | 4            
Male   | Common Cold           | Adult     | 4            
Male   | Diabetes              | Senior    | 4            
Female | Ear Infection         | Pediatric | 4            
Male   | Muscle Injury         | Sen

In [ ]:
--What are the top 5 most common diagnoses?--

SELECT TOP 5
diagnosis,
COUNT(*) AS total_patient_count
FROM [Healthcare_Database].[dbo].[Outpatient Visits]
WHERE diagnosis <> 'Unknown'
GROUP BY diagnosis
ORDER BY total_patient_count DESC

(5 rows affected)

diagnosis           | total_patient_count
--------------------+--------------------
Diabetes            | 37                 
Respiratory Illness | 26                 
Muscle Injury       | 24                 
Hypertension        | 23                 
Common Cold         | 13                 
(5 rows)

Total execution time: 00:00:00.003

In [ ]:
--When do appointments occur most often across the day?--
SELECT
DATEPART (hour, appointment_time) AS appointment_hour,
COUNT(*) AS appointment_count
FROM [Healthcare_Database].[dbo].Appointments
GROUP BY
DATEPART (hour, appointment_time)
ORDER BY appointment_count DESC

(5 rows affected)

appointment_hour | appointment_count
-----------------+------------------
12               | 46               
9                | 40               
11               | 38               
10               | 34               
8                | 16               
(5 rows)

Total execution time: 00:00:00.004

In [ ]:
--Which lab tests are ordered most frequently?
SELECT
test_name,
COUNT(*) AS test_count
FROM [Healthcare_Database].[dbo].[Lab Results]
GROUP BY test_name
ORDER BY test_count DESC

(10 rows affected)

test_name                   | test_count
----------------------------+-----------
Chloride                    | 49        
Fasting Blood Sugar         | 42        
ALT                         | 40        
Thyroid Stimulating Hormone | 40        
White Blood Cells           | 39        
Uric Acid                   | 37        
Hemoglobin A1C              | 35        
C-Reactive Protein          | 35        
HDL Cholesterol             | 34        
Creatinine                  | 29        
(10 rows)

Total execution time: 00:00:00.015

In [ ]:
--Which patients have fasting blood sugar results outside the normal range of 70-100 mg per dL?--
SELECT
p.patient_id,
p.patient_name,
result_value
FROM [Healthcare_Database].[dbo].Patients AS p
INNER JOIN [Healthcare_Database].[dbo].[Outpatient Visits] AS ov
ON p.patient_id = ov.patient_id
INNER JOIN [Healthcare_Database].[dbo].[Lab Results] AS lr
ON ov.visit_id = lr.visit_id
WHERE lr.test_name = 'Fasting Blood Sugar'
AND (lr.result_value < 70 OR lr.result_value >100)

(21 rows affected)

patient_id | patient_name    | result_value
-----------+-----------------+-------------
521001     | Emma Johnson    | 107         
521001     | Emma Johnson    | 101         
521002     | Michael Smith   | 103         
521004     | William Jones   | 110         
521004     | William Jones   | 105.82      
521008     | Bethany Clark   | 58.95       
521009     | Jose Gonzalez   | 110         
521011     | Amelia Thomas   | 111.47      
521011     | Amelia Thomas   | 112         
521013     | Alexander Perez | 55.87       
521021     | Olivia Scott    | 107.99      
521024     | Liam Baker      | 124.81      
521030     | James Cooper    | 66.38       
521031     | Andy Morris     | 127         
521032     | Rick Sanders    | 50.91       
521033     | Eva Torres      | 123.27      
521047     | Nick Reid       | 55.34       
521069     | Sara Martin     | 120         
521074     | Sophie Myers    | 105         
521092     | Ava White       | 110         
521092     |

In [ ]:
--How many patients fall into each risk category?--

SELECT
CASE
WHEN smoker_status = 'Y' AND (diagnosis = 'Hypertension' OR diagnosis = 'Diabetes') THEN 'High Risk'
WHEN smoker_status = 'N' AND (diagnosis = 'Hypertension' OR diagnosis = 'Diabetes') THEN 'Medium Risk'
ELSE 'Low Risk'
END AS Risk_category,
COUNT(patient_id) AS num_patients
FROM [Healthcare_Database].[dbo].[Outpatient Visits]
GROUP BY
CASE
WHEN smoker_status = 'Y' AND (diagnosis = 'Hypertension' OR diagnosis = 'Diabetes') THEN 'High Risk'
WHEN smoker_status = 'N' AND (diagnosis = 'Hypertension' OR diagnosis = 'Diabetes') THEN 'Medium Risk'
ELSE 'Low Risk'
END

(3 rows affected)

Risk_category | num_patients
--------------+-------------
High Risk     | 21          
Low Risk      | 389         
Medium Risk   | 39          
(3 rows)

Total execution time: 00:00:00.017

In [ ]:
--What are the appointment patterns across the day?--
SELECT
DATEPART (hour, appointment_time) AS appointment_hour,
COUNT(*) AS appointment_count
FROM [Healthcare_Database].[dbo].Appointments
GROUP BY
DATEPART (hour, appointment_time)
ORDER BY appointment_count DESC

(5 rows affected)

appointment_hour | appointment_count
-----------------+------------------
12               | 46               
9                | 40               
11               | 38               
10               | 34               
8                | 16               
(5 rows)

Total execution time: 00:00:00.009

In [21]:
--Which patients returned for another visit within 30 days?--

SELECT
ov_initial.patient_id,
ov_initial.visit_date AS initial_visit_date,
ov_initial.reason_for_visit AS reason_for_initial_visit,
ov_readmit.visit_date AS readmission_date,
ov_readmit.reason_for_visit AS reason_for_readmission,
DATEDIFF(day, ov_initial.visit_date, ov_readmit.visit_date) AS days_between
FROM 
[Healthcare_Database].[dbo].[Outpatient Visits] AS ov_initial
INNER JOIN [Healthcare_Database].[dbo].[Outpatient Visits] AS ov_readmit
ON ov_initial.patient_id = ov_readmit.patient_id
WHERE DATEDIFF(day, ov_initial.visit_date, ov_readmit.visit_date) <= 30
AND ov_readmit.visit_date > ov_initial.visit_date

(73 rows affected)

patient_id | initial_visit_date      | reason_for_initial_visit     | readmission_date        | reason_for_readmission       | days_between
-----------+-------------------------+------------------------------+-------------------------+------------------------------+-------------
521025     | 2024-12-07 00:00:00.000 | Checkup                      | 2024-12-31 00:00:00.000 | Checkup                      | 24          
521026     | 2024-05-31 00:00:00.000 | Knee pain                    | 2024-06-10 00:00:00.000 | Checkup                      | 10          
521027     | 2024-10-10 00:00:00.000 | Diabetes check               | 2024-11-08 00:00:00.000 | Annual physical              | 29          
521027     | 2024-11-08 00:00:00.000 | Annual physical              | 2024-11-10 00:00:00.000 | Diet and Exercise Counseling | 2           
521028     | 2023-05-29 00:00:00.000 | Fever                        | 2023-06-15 00:00:00.000 | Annual physical              | 17          
